# ToW Reproduction — Repo-Aligned Colab Runner

This notebook is for the **recreation/reproduction** part only. It keeps the official `ARC-ASU/fine-nwp` repo as the source. Training calls `fine_nwp/finetune.py`, evaluation calls `python -m eval.run_eval`, and the Raw/ToW conditions use the repo's prepared `data/ToW-pretrain/*.jsonl` files.

This is still a **Colab/A100-safe scaled reproduction**: the original paper used 7B/8B-class models and larger effective batch sizes, while this default uses `Qwen/Qwen2.5-3B` so the whole Raw vs. ToW comparison can finish reliably on a single A100 runtime.

## Repo/paper alignment

This notebook mirrors the original structure:

- **Raw condition:** `data/ToW-pretrain/Raw.jsonl`
- **ToW condition:** `data/ToW-pretrain/ToW.jsonl`
- **Trainer:** official `fine_nwp/finetune.py`
- **Evaluator:** official `eval/run_eval.py`
- **Metric summary:** reads each benchmark's `metrics.json`

The paper used 6K documents from OpenWebMath/C4, about 8M tokens, 70K+ ToW annotations, causal-LM training, and exact-match evaluation on reasoning/hallucination benchmarks. This notebook uses the same repo data and scripts, but defaults to a smaller model/checkpoint plan for Colab stability.

I could not inspect any private Discord content from the project. If you have exported Discord notes/logs, upload them and this notebook can be adjusted to match those too.

## 0. Settings — time-limited report run

This notebook is configured for a **fast, transparent scaled reproduction**. It keeps the official `fine-nwp` training/eval code path and Raw-vs-ToW comparison, but uses a 3B model, 30 training update steps, two benchmarks, and a capped eval size so it can finish in roughly 30 minutes on an A100.

For the paper, describe this as a **time/resource-limited reproduction**, not a full replication of all models/benchmarks from the original paper.


In [1]:
# ===== USER SETTINGS =====
# Start with "smoke". After Base smoke eval works, run Raw/ToW in "pilot" or "full".
RUN_MODE = "pilot"   # options: "smoke", "pilot", "full"

# Paper models were 7B/8B-class: Mistral-7B, LLaMA2-7B, LLaMA3-8B, Qwen2.5-7B, Falcon-7B.
# Colab-safe default: Qwen2.5-3B. The helper scripts also support mistral_7b and llama3_8b if your HF access/runtime allows.
MODEL_KEYS = ["qwen2_5_3b"]

# Paper benchmarks: gsm8k, csqa, strategyqa, arc-challenge, truthfulqa, halueval.
# Keep ARC first because it is the quickest sanity check and matches the science-reasoning results.
BENCHMARKS = ["arc-challenge", "csqa"]
CONDITIONS = ["Raw", "ToW"]

# Smoke/pilot/full sizes.
# Time-limited report run: pilot uses 30 update steps and 50 eval examples per benchmark.
# This gives Raw-vs-ToW results on ARC-Challenge and CSQA while staying practical in Colab.
# The original paper used a broader/larger setup, so report this as a scaled reproduction.
PILOT_MAX_TRAIN_STEPS = 30
PILOT_MAX_TRAIN_SAMPLES = 1024
PILOT_NUM_EVAL_INSTANCES = 50

FULL_MAX_TRAIN_STEPS = 30
FULL_MAX_TRAIN_SAMPLES = ""   # blank = all examples in the repo JSONL
FULL_NUM_EVAL_INSTANCES = ""  # blank = full eval split

# A100 80GB + High-RAM-safe defaults for the 3B model.
# The paper uses larger effective batch sizes; this keeps the same code path but fits Colab reliably.
TOTAL_BATCH_SIZE = 4
BATCH_SIZE_PER_GPU = 1
MAX_SEQ_LENGTH = 768

# Use no-offload for 3B on A100 to avoid saturating system RAM.
# If you hit CUDA OOM, switch to "configs/stage3_offloading_accelerate.conf" and/or set MAX_SEQ_LENGTH = 768 or 512.
DEEPSPEED_CONFIG_FILE = "configs/stage3_no_offloading_accelerate.conf"
MIN_CHECKPOINT_MB = 500

PROJECT_DIR = "/content/drive/MyDrive/tow_project"
REPO_DIR = "/content/tow"
REPO_URL = "https://github.com/ARC-ASU/fine-nwp"

# Heavy checkpoint behavior.
# Training writes to fast local disk first, then syncs to Drive after training finishes.
LOCAL_CHECKPOINT_ROOT = "/content/tow_runtime/checkpoints"
LOCAL_TORCH_EXTENSIONS = "/content/tow_runtime/torch_extensions"
SYNC_CHECKPOINT_TO_DRIVE = "1"
KEEP_LOCAL_CHECKPOINTS = "1"

# Set True only if you want the install cell to restart Colab automatically after pip changes.
AUTO_RESTART_AFTER_INSTALL = False

print("RUN_MODE:", RUN_MODE)
print("MODEL_KEYS:", MODEL_KEYS)
print("BENCHMARKS:", BENCHMARKS)
print("CONDITIONS:", CONDITIONS)
print("FULL_MAX_TRAIN_STEPS:", FULL_MAX_TRAIN_STEPS)
print("TOTAL_BATCH_SIZE:", TOTAL_BATCH_SIZE)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("DEEPSPEED_CONFIG_FILE:", DEEPSPEED_CONFIG_FILE)
print("LOCAL_CHECKPOINT_ROOT:", LOCAL_CHECKPOINT_ROOT)

RUN_MODE: pilot
MODEL_KEYS: ['qwen2_5_3b']
BENCHMARKS: ['arc-challenge', 'csqa']
CONDITIONS: ['Raw', 'ToW']
FULL_MAX_TRAIN_STEPS: 30
TOTAL_BATCH_SIZE: 4
MAX_SEQ_LENGTH: 768
DEEPSPEED_CONFIG_FILE: configs/stage3_no_offloading_accelerate.conf
LOCAL_CHECKPOINT_ROOT: /content/tow_runtime/checkpoints


### What this run can support in the paper

Use these results to support a **scaled reproduction claim**: same official code/data path, same Raw vs. ToW comparison, smaller model/run budget. Do not claim this fully reproduces every table in the original paper.

Recommended wording: “Due to Colab time limits, I performed a scaled reproduction using Qwen2.5-3B for 30 update steps on ARC-Challenge. The goal was to verify whether the released Raw/ToW pipeline could be executed and whether the direction of Raw-vs-ToW effects could be observed under a smaller budget.”


## 1. Mount Drive and verify runtime


In [2]:
from google.colab import drive
import os, json, pathlib, subprocess, sys

drive.mount('/content/drive')
for sub in ["logs", "results", "checkpoints", "scripts", "notebooks", "extension"]:
    os.makedirs(f"{PROJECT_DIR}/{sub}", exist_ok=True)
os.makedirs(LOCAL_CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(LOCAL_TORCH_EXTENSIONS, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Local checkpoint root:", LOCAL_CHECKPOINT_ROOT)
print("Run mode:", RUN_MODE)
print("Models:", MODEL_KEYS)
print("Benchmarks:", BENCHMARKS)
print("Conditions:", CONDITIONS)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/tow_project
Local checkpoint root: /content/tow_runtime/checkpoints
Run mode: pilot
Models: ['qwen2_5_3b']
Benchmarks: ['arc-challenge', 'csqa']
Conditions: ['Raw', 'ToW']


In [3]:
# No checkpoint check here. Checkpoints are verified after training in the reproduction section.
print("Setup cell complete.")


Setup cell complete.


In [4]:
!nvidia-smi
print("\nSystem RAM:")
!free -h
print("\nColab note: for full-parameter 7B training, use Runtime → Change runtime type → High-RAM if available.")

Mon May  4 20:30:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone repo locally

In [5]:
%cd /content
import os, subprocess, pathlib, shutil

FORCE_RECLONE = True  # keep True for a clean project run; set False if you want to keep local edits in /content/tow

if FORCE_RECLONE and os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, "tow"], check=True)
else:
    print("Repo already exists:", REPO_DIR)

%cd {REPO_DIR}
!pwd
!ls


/content
/content/tow
/content/tow
assets	configs  data  eval  fine_nwp  README.md  requirements.txt  scripts


## 3. Install dependencies

This cell skips `flash-attn`, pins the repo-friendly dependency stack, and avoids the package drift that caused `numpy==2.x`, newer Transformers, too-new PEFT, and vLLM rope-scaling failures.

**Important Colab rule:** after the first dependency install in a fresh runtime, restart the runtime once before continuing. This cell writes a versioned marker at `/content/.tow_deps_installed_locked_v5`, so after restart it will skip reinstalling packages in the same Colab VM.

If the environment ever breaks again, delete the marker and rerun this cell:

```bash
!rm -f /content/.tow_deps_installed_locked_v*
```



PEFT is pinned to `0.12.0` because newer PEFT imports `HybridCache`, which is not available in `transformers==4.43.4`. `huggingface-hub` is pinned to `0.24.7` because Transformers 4.43.4 requires `<1.0`. vLLM is installed once with dependencies, then fragile packages are re-pinned afterward; this prevents missing modules like `msgspec` while still avoiding the NumPy/Transformers drift that broke evaluation.


In [6]:
%cd {REPO_DIR}

import os, sys, subprocess, pathlib, textwrap

# Versioned marker. Changing the marker name forces a clean reinstall when we update the dependency strategy.
# v7 installs vLLM with dependencies first, re-pins fragile packages, and supports the repo-aligned reproduction runner.
marker = pathlib.Path("/content/.tow_deps_installed_locked_v7")
req_path = pathlib.Path("requirements.txt")
colab_req = pathlib.Path("requirements_colab.txt")

TARGETS = {
    "numpy": "1.26.4",
    "transformers": "4.43.4",
    "tokenizers": "0.19.1",
    "vllm": "0.6.1.post2",
    "peft": "0.12.0",
    "huggingface_hub": "0.24.7",
}

def run(cmd):
    print("\n>>>", " ".join(cmd))
    subprocess.run(cmd, check=True)

def version_ok():
    try:
        import numpy as np
        import transformers, tokenizers, vllm, peft, huggingface_hub
        import msgspec  # vLLM runtime dependency; missing this caused the prior reinstall loop.
        ok = (
            np.__version__ == TARGETS["numpy"] and
            transformers.__version__ == TARGETS["transformers"] and
            tokenizers.__version__ == TARGETS["tokenizers"] and
            getattr(vllm, "__version__", "") == TARGETS["vllm"] and
            peft.__version__ == TARGETS["peft"] and
            huggingface_hub.__version__ == TARGETS["huggingface_hub"]
        )
        print("Current dependency versions:")
        print("  numpy:", np.__version__)
        print("  transformers:", transformers.__version__)
        print("  tokenizers:", tokenizers.__version__)
        print("  vllm:", getattr(vllm, "__version__", "unknown"))
        print("  peft:", peft.__version__)
        print("  huggingface_hub:", huggingface_hub.__version__)
        print("  msgspec:", getattr(msgspec, "__version__", "imported"))
        return ok
    except Exception as e:
        print("Version check failed before install:", repr(e))
        return False

if marker.exists() and version_ok():
    print("Dependency marker found and versions look correct:", marker)
    print("Skipping pip install for this runtime.")
else:
    # Delete all old markers so stale installs do not get reused.
    for old in pathlib.Path("/content").glob(".tow_deps_installed_locked_v*"):
        old.unlink(missing_ok=True)

    # Filter out flash-attn because building it from source often stalls on Colab.
    # Also let our final pins control fragile packages that caused prior failures.
    lines = req_path.read_text().splitlines()
    filtered = []
    fragile_prefixes = (
        "transformers", "tokenizers", "numpy", "protobuf", "peft", "accelerate",
        "vllm", "fsspec", "requests", "huggingface-hub", "huggingface_hub", "torch", "torchaudio", "torchvision"
    )
    for line in lines:
        low = line.strip().lower()
        if not low or low.startswith("#"):
            filtered.append(line)
            continue
        if "flash-attn" in low:
            continue
        if any(low.startswith(prefix) for prefix in fragile_prefixes):
            continue
        filtered.append(line)
    colab_req.write_text("\n".join(filtered) + "\n")
    print("Wrote:", colab_req)

    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])

    # Install repo requirements first, minus fragile packages.
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(colab_req)])

    # Install vLLM WITH dependencies so runtime deps like msgspec/outlines/xformers are present.
    # This may temporarily upgrade fragile packages; the hard pins below fix them afterward.
    run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir",
         "vllm==0.6.1.post2"])

    # Install/upgrade support packages.
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-cache-dir",
         "accelerate>=1.0.0,<2.0.0", "bitsandbytes", "evaluate", "scikit-learn", "sentencepiece", "psutil"])

    # Final hard pins. Keep these AFTER vLLM so pip cannot leave NumPy/Transformers/HF Hub in incompatible versions.
    run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir",
         "numpy==1.26.4", "protobuf==5.29.5", "requests==2.32.4", "fsspec==2025.3.0", "huggingface-hub==0.24.7"])
    run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir", "--no-deps",
         "transformers==4.43.4", "tokenizers==0.19.1", "peft==0.12.0"])

    # Sanity import in a fresh subprocess. This catches missing modules like msgspec before we write the marker.
    sanity_code = """
import numpy as np
import transformers, tokenizers, vllm, peft, huggingface_hub, msgspec
assert np.__version__ == '1.26.4', np.__version__
assert transformers.__version__ == '4.43.4', transformers.__version__
assert tokenizers.__version__ == '0.19.1', tokenizers.__version__
assert peft.__version__ == '0.12.0', peft.__version__
assert huggingface_hub.__version__ == '0.24.7', huggingface_hub.__version__
print('dependency sanity ok')
"""
    run([sys.executable, "-c", sanity_code])

    marker.write_text("locked-v7-repo-aligned-reproduction\n")
    print("\nDependencies installed with locked versions and vLLM deps.")
    print("RESTART RUNTIME NOW: Runtime → Restart runtime.")
    print("After restart, rerun from Settings through this cell; this cell should skip reinstalling if versions are correct.")
    if AUTO_RESTART_AFTER_INSTALL:
        import os as _os
        _os.kill(_os.getpid(), 9)
    # Do not raise an exception here. Stop manually and restart once.
    # If you keep running without restarting, imports may still use stale binary packages.


/content/tow
Current dependency versions:
  numpy: 1.26.4
  transformers: 4.43.4
  tokenizers: 0.19.1
  vllm: 0.6.1.post2
  peft: 0.12.0
  huggingface_hub: 0.24.7
  msgspec: 0.21.1
Dependency marker found and versions look correct: /content/.tow_deps_installed_locked_v7
Skipping pip install for this runtime.


In [7]:
# Run this only AFTER restarting the runtime once after dependency installation.
import sys
try:
    import numpy as np
    import torch
    import transformers
    import tokenizers
    import datasets
    import accelerate
    import evaluate
    import vllm
    import deepspeed
    import peft
    import huggingface_hub
    import msgspec
    from google.protobuf import __version__ as protobuf_version

    print("python:", sys.version.split()[0])
    print("numpy:", np.__version__)
    print("torch:", torch.__version__)
    print("cuda:", torch.cuda.is_available())
    print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")
    print("transformers:", transformers.__version__)
    print("tokenizers:", tokenizers.__version__)
    print("datasets:", datasets.__version__)
    print("accelerate:", accelerate.__version__)
    print("peft:", peft.__version__)
    print("huggingface_hub:", huggingface_hub.__version__)
    print("vllm:", vllm.__version__ if hasattr(vllm, "__version__") else "imported")
    print("deepspeed:", deepspeed.__version__ if hasattr(deepspeed, "__version__") else "imported")
    print("protobuf:", protobuf_version)
    print("msgspec:", getattr(msgspec, "__version__", "imported"))

    assert np.__version__ == "1.26.4", "NumPy must stay at 1.26.4 for vLLM/outlines. Rerun dependency cell."
    assert transformers.__version__ == "4.43.4", "Transformers must stay at 4.43.4 for this repo/vLLM stack."
    assert tokenizers.__version__ == "0.19.1", "Tokenizers must stay at 0.19.1."
    assert peft.__version__ == "0.12.0", "PEFT must stay at 0.12.0 to avoid HybridCache import errors with Transformers 4.43.4."
    assert huggingface_hub.__version__ == "0.24.7", "huggingface-hub must stay at 0.24.7 because Transformers 4.43.4 requires <1.0."
    assert getattr(vllm, "__version__", "") == "0.6.1.post2", "vLLM must stay at 0.6.1.post2."

    print("imports and version pins ok")
except Exception as e:
    print("Import/version test failed.")
    print("Most common fix: delete /content/.tow_deps_installed_locked_v6, rerun the dependency cell, restart runtime, then rerun setup.")
    print("Error:", repr(e))
    raise


[2026-05-04 20:31:38,803] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
python: 3.12.13
numpy: 1.26.4
torch: 2.4.0+cu121
cuda: True
gpu: NVIDIA A100-SXM4-80GB
transformers: 4.43.4
tokenizers: 0.19.1
datasets: 4.8.5
accelerate: 1.13.0
peft: 0.12.0
huggingface_hub: 0.24.7
vllm: 0.6.1.post2
deepspeed: 0.15.0
protobuf: 5.29.5
msgspec: 0.21.1
imports and version pins ok


## 4. Hugging Face login

Run this if your chosen model requires Hugging Face access. The default `qwen2_5_3b` model is normally not gated, but logging in is still fine.


In [8]:
from huggingface_hub import login
login()


## 5. Patch repo for Colab

This cell fixes the two issues that kept breaking the run:

1. It disables FlashAttention-specific assumptions in the finetuning code.
2. It patches `eval/run_eval.py` so `--num_instances` works when the loaded eval data is a normal Python list instead of a pandas DataFrame.

Run this cell after cloning the repo. It is safe to rerun; it resets the target files from git first, then reapplies the patches.


In [9]:
%cd {REPO_DIR}

from pathlib import Path
import re, subprocess, sys, os

# Start from clean repo versions so rerunning this cell cannot stack broken patches.
subprocess.run(
    "git checkout -- eval/run_eval.py fine_nwp/finetune.py fine_nwp/model_utils.py 2>/dev/null || true",
    shell=True,
    check=False,
)

# ------------------------------------------------------------
# Patch 1: avoid FlashAttention-specific settings when possible
# ------------------------------------------------------------
flash_replacements = {
    'attn_implementation="flash_attention_2"': 'attn_implementation="sdpa"',
    "attn_implementation='flash_attention_2'": "attn_implementation='sdpa'",
    '"flash_attention_2"': '"sdpa"',
    "'flash_attention_2'": "'sdpa'",
    "use_flash_attention_2=True": "use_flash_attention_2=False",
    "use_flash_attn=True": "use_flash_attn=False",
}

changed_files = []
for fp in list(Path("fine_nwp").glob("*.py")) + [Path("eval/run_eval.py")]:
    if not fp.exists():
        continue
    text = fp.read_text()
    old_text = text
    for old, new in flash_replacements.items():
        text = text.replace(old, new)
    if text != old_text:
        fp.write_text(text)
        changed_files.append(str(fp))

print("FlashAttention patch files:", changed_files if changed_files else "no direct flash-attn strings found")

# ------------------------------------------------------------
# Patch 2: make --num_instances work for both DataFrame and list data
# ------------------------------------------------------------
p = Path("eval/run_eval.py")
text = p.read_text()

# Ensure global random import exists. Do not import random inside main(),
# because that causes UnboundLocalError when random.seed(42) appears earlier.
if not re.search(r"(?m)^\s*import\s+random\b", text):
    lines = text.splitlines()
    insert_at = 0
    for i, line in enumerate(lines):
        if line.startswith("import ") or line.startswith("from "):
            insert_at = i + 1
    lines.insert(insert_at, "import random")
    text = "\n".join(lines) + "\n"

lines = text.splitlines()
idx = None
for i, line in enumerate(lines):
    if "if args.num_instances is not None" in line:
        idx = i
        break

if idx is None:
    raise RuntimeError("Could not find the 'if args.num_instances is not None' block in eval/run_eval.py")

indent = lines[idx][:len(lines[idx]) - len(lines[idx].lstrip())]
base_indent_len = len(indent)

replacement = [
    f"{indent}if args.num_instances is not None:",
    f"{indent}    _n = min(args.num_instances, len(all_data))",
    f"{indent}    if hasattr(all_data, 'sample'):",
    f"{indent}        all_data = all_data.sample(_n, random_state=42)",
    f"{indent}    else:",
    f"{indent}        all_data = random.sample(all_data, _n)",
]

# Remove the original block body. This also cleans up earlier failed patch attempts.
end = idx + 1
while end < len(lines):
    s = lines[end].strip()
    if s == "":
        end += 1
        continue
    curr_indent = len(lines[end]) - len(lines[end].lstrip())
    if curr_indent > base_indent_len:
        end += 1
        continue
    break

new_lines = lines[:idx] + replacement + lines[end:]
p.write_text("\n".join(new_lines) + "\n")

print("\nPatched eval/run_eval.py num_instances block:")
patched = p.read_text().splitlines()
for n in range(max(1, idx - 4), min(len(patched), idx + 12) + 1):
    print(f"{n:03d}: {patched[n-1]}")

# ------------------------------------------------------------
# Patch 3: make fine_nwp/finetune.py compatible with current Colab libraries
# ------------------------------------------------------------
fp = Path("fine_nwp/finetune.py")
text = fp.read_text()

# 3a. New accelerate no longer accepts use_seedable_sampler in Accelerator(...).
lines = text.splitlines()
out = []
inside_accelerator = False
depth = 0

for line in lines:
    if not inside_accelerator and "Accelerator(" in line:
        inside_accelerator = True
        depth = line.count("(") - line.count(")")
    elif inside_accelerator:
        depth += line.count("(") - line.count(")")

    if inside_accelerator and "use_seedable_sampler" in line:
        indent = line[:len(line) - len(line.lstrip())]
        out.append(indent + "# " + line.lstrip() + "  # patched out for Colab accelerate compatibility")
    else:
        out.append(line)

    if inside_accelerator and depth <= 0:
        inside_accelerator = False

text = "\n".join(out) + "\n"

# 3b. DeepSpeed ZeRO-3 is incompatible with low_cpu_mem_usage=True and device_map.
text = re.sub(
    r"low_cpu_mem_usage\s*=\s*[^,\n\)]*",
    "low_cpu_mem_usage=False",
    text,
)

# Comment out device_map lines instead of deleting them, so function-call syntax remains stable.
patched_lines = []
for line in text.splitlines():
    stripped = line.strip()
    if stripped.startswith("device_map") and "=" in stripped:
        indent = line[:len(line) - len(line.lstrip())]
        patched_lines.append(indent + "# " + stripped + "  # patched out for DeepSpeed ZeRO-3 compatibility")
    else:
        patched_lines.append(line)
text = "\n".join(patched_lines) + "\n"

# 3c. Suppress huge tokenized sample logs without leaving an empty block.
patched_lines = []
for line in text.splitlines():
    if ("logger.info" in line and "Sample" in line and "training set" in line):
        indent = line[:len(line) - len(line.lstrip())]
        patched_lines.append(indent + "pass  # patched out logger.info sample dump to reduce Colab log spam")
    else:
        patched_lines.append(line)
text = "\n".join(patched_lines) + "\n"

fp.write_text(text)

print("\nFine-tune compatibility patch summary:")
print("use_seedable_sampler lines:")
subprocess.run("grep -n 'use_seedable_sampler' fine_nwp/finetune.py || true", shell=True, check=False)

print("\nlow_cpu_mem_usage/device_map lines:")
subprocess.run("grep -n 'low_cpu_mem_usage\\|device_map' fine_nwp/finetune.py || true", shell=True, check=False)

# ------------------------------------------------------------
# Syntax checks, with useful error output
# ------------------------------------------------------------
print("\nSyntax checks:")

for target in ["eval/run_eval.py", "fine_nwp/finetune.py"]:
    result = subprocess.run(
        [sys.executable, "-m", "py_compile", target],
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        print(f"\nSYNTAX CHECK FAILED: {target}")
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"Syntax check failed for {target}")
    print(f"OK: {target}")

print("Repo patch complete.")

/content/tow
FlashAttention patch files: ['fine_nwp/finetune.py', 'fine_nwp/model_utils.py']

Patched eval/run_eval.py num_instances block:
085:         all_data = []
086:         with open(os.path.join(args.data_dir, "test.jsonl")) as f:
087:             for line in f.readlines():
088:                 all_data.append(json.loads(line))
089: 
090:         if args.num_instances is not None:
091:             _n = min(args.num_instances, len(all_data))
092:             if hasattr(all_data, 'sample'):
093:                 all_data = all_data.sample(_n, random_state=42)
094:             else:
095:                 all_data = random.sample(all_data, _n)
096:         prompts = []
097:         targets = []
098:         for data in all_data:
099:             question = data['question']['stem'].strip()
100: 
101:             choices = ''

Fine-tune compatibility patch summary:
use_seedable_sampler lines:

low_cpu_mem_usage/device_map lines:

Syntax checks:
OK: eval/run_eval.py
OK: fine_nwp/finetun

## 6. Check data and scripts

In [10]:
%cd {REPO_DIR}

print("Pretrain data:")
!ls -lh data/ToW-pretrain || true

print("\nEval data directories:")
!find data/eval -maxdepth 2 -type d | sort || true

print("\nEval help:")
!python -m eval.run_eval --help | head -80


/content/tow
Pretrain data:
total 216M
-rw-r--r-- 1 root root 28M May  4 20:31 Raw.jsonl
-rw-r--r-- 1 root root 30M May  4 20:31 ToW-em_only.jsonl
-rw-r--r-- 1 root root 34M May  4 20:31 ToW.jsonl
-rw-r--r-- 1 root root 53M May  4 20:31 ToW-NoDeN.jsonl
-rw-r--r-- 1 root root 34M May  4 20:31 ToW-no_unpred.jsonl
-rw-r--r-- 1 root root 40M May  4 20:31 ToW-PartDeN.jsonl

Eval data directories:
data/eval
data/eval/arc_challenge
data/eval/csqa
data/eval/gsm
data/eval/halueval
data/eval/strategyqa
data/eval/truthfulqa

Eval help:
usage: run_eval.py [-h] [--model_name_or_path MODEL_NAME_OR_PATH]
                   [--hf_revision HF_REVISION]
                   [--tokenizer_name_or_path TOKENIZER_NAME_OR_PATH]
                   [--use_slow_tokenizer] [--data_dir DATA_DIR]
                   [--save_dir SAVE_DIR] [--num_instances NUM_INSTANCES]
                   [--newline_stop] [--stop_at_double_newline]
                   [--stop_at_triple_newline]
                   [--additional_stop_seq

## 7. Write helper scripts


In [11]:
%%writefile /content/drive/MyDrive/tow_project/scripts/tow_train_one.sh
#!/usr/bin/env bash
set -euo pipefail

# Colab/DeepSpeed compatibility.
export DS_SKIP_CUDA_CHECK=${DS_SKIP_CUDA_CHECK:-1}
export MAX_JOBS=${MAX_JOBS:-2}

MODEL_KEY=${1:?MODEL_KEY required: qwen2_5_3b, phi3_mini, tinyllama_1b, mistral_7b, or llama3_8b}
COND=${2:?COND required: Raw or ToW}
# Third arg can be a number of steps or a run mode string.
THIRD=${3:-${RUN_MODE:-smoke}}

PROJECT_DIR=${PROJECT_DIR:-/content/drive/MyDrive/tow_project}
REPO_DIR=${REPO_DIR:-/content/tow}
LOCAL_CHECKPOINT_ROOT=${LOCAL_CHECKPOINT_ROOT:-/content/tow_runtime/checkpoints}
LOCAL_TORCH_EXTENSIONS=${LOCAL_TORCH_EXTENSIONS:-/content/tow_runtime/torch_extensions}
SYNC_CHECKPOINT_TO_DRIVE=${SYNC_CHECKPOINT_TO_DRIVE:-1}
KEEP_LOCAL_CHECKPOINTS=${KEEP_LOCAL_CHECKPOINTS:-1}
MIN_CHECKPOINT_MB=${MIN_CHECKPOINT_MB:-500}

export TORCH_EXTENSIONS_DIR=${TORCH_EXTENSIONS_DIR:-$LOCAL_TORCH_EXTENSIONS}
mkdir -p "$TORCH_EXTENSIONS_DIR" "$LOCAL_CHECKPOINT_ROOT" "$PROJECT_DIR/logs" "$PROJECT_DIR/checkpoints"

echo "=== RUNTIME RESOURCE CHECK ==="
nvidia-smi || true
free -h || true
df -h /content || true
echo "=============================="

TOTAL_BATCH_SIZE=${TOTAL_BATCH_SIZE:-4}
BATCH_SIZE_PER_GPU=${BATCH_SIZE_PER_GPU:-1}
MAX_SEQ_LENGTH=${MAX_SEQ_LENGTH:-1024}
MAX_TRAIN_SAMPLES=${MAX_TRAIN_SAMPLES:-}
# For the default 3B model on A100, no-offload is safer because it avoids filling 167GB system RAM.
DEEPSPEED_CONFIG_FILE=${DEEPSPEED_CONFIG_FILE:-configs/stage3_no_offloading_accelerate.conf}

if [[ "$THIRD" =~ ^[0-9]+$ ]]; then
  MAX_TRAIN_STEPS="$THIRD"
else
  RUN_MODE="$THIRD"
  MAX_TRAIN_STEPS=${MAX_TRAIN_STEPS:-2}
fi

cd "$REPO_DIR"

TOKENIZER_FLAGS=""
case "$MODEL_KEY" in
  qwen2_5_3b) MODEL_NAME="Qwen/Qwen2.5-3B" ;;
  phi3_mini) MODEL_NAME="microsoft/Phi-3-mini-4k-instruct" ;;
  tinyllama_1b) MODEL_NAME="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T" ;;
  mistral_7b) MODEL_NAME="mistralai/Mistral-7B-v0.1"; TOKENIZER_FLAGS="--use_slow_tokenizer" ;;
  llama3_8b) MODEL_NAME="meta-llama/Meta-Llama-3-8B" ;;
  *) echo "Unknown MODEL_KEY: $MODEL_KEY"; exit 1 ;;
esac

case "$COND" in
  Raw) TRAIN_FILE="data/ToW-pretrain/Raw.jsonl" ;;
  ToW) TRAIN_FILE="data/ToW-pretrain/ToW.jsonl" ;;
  ToW-NoDeN) TRAIN_FILE="data/ToW-pretrain/ToW-NoDeN.jsonl" ;;
  ToW-PartDeN) TRAIN_FILE="data/ToW-pretrain/ToW-PartDeN.jsonl" ;;
  *) echo "Unknown condition: $COND"; exit 1 ;;
esac

if [ ! -f "$TRAIN_FILE" ]; then
  echo "ERROR: training file not found: $TRAIN_FILE"
  exit 2
fi

LOCAL_OUT="$LOCAL_CHECKPOINT_ROOT/$MODEL_KEY/$COND"
DRIVE_OUT="$PROJECT_DIR/checkpoints/$MODEL_KEY/$COND"
rm -rf "$LOCAL_OUT"
mkdir -p "$LOCAL_OUT" "$(dirname "$DRIVE_OUT")"

NUM_GPUS=$(python -c "import torch; print(max(1, torch.cuda.device_count()))")
PRECISION=$(python -c "import torch; print('bf16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'fp16')")
echo "PRECISION=$PRECISION (auto-detected; T4/V100 -> fp16, A100/L4/H100 -> bf16)"

GRADIENT_ACC_STEPS=$(( TOTAL_BATCH_SIZE / NUM_GPUS / BATCH_SIZE_PER_GPU ))
if [ "$GRADIENT_ACC_STEPS" -lt 1 ]; then GRADIENT_ACC_STEPS=1; fi

EXTRA_ARGS=""
if [ -n "$MAX_TRAIN_SAMPLES" ]; then
  if python fine_nwp/finetune.py --help 2>&1 | grep -q -- "--max_train_samples"; then
    EXTRA_ARGS="$EXTRA_ARGS --max_train_samples $MAX_TRAIN_SAMPLES"
  else
    echo "WARNING: finetune.py does not expose --max_train_samples; ignoring MAX_TRAIN_SAMPLES=$MAX_TRAIN_SAMPLES"
  fi
fi

echo "=== TRAIN START ==="
echo "MODEL_KEY=$MODEL_KEY"
echo "MODEL_NAME=$MODEL_NAME"
echo "COND=$COND"
echo "TRAIN_FILE=$TRAIN_FILE"
echo "LOCAL_OUT=$LOCAL_OUT"
echo "DRIVE_OUT=$DRIVE_OUT"
echo "NUM_GPUS=$NUM_GPUS"
echo "TOTAL_BATCH_SIZE=$TOTAL_BATCH_SIZE"
echo "BATCH_SIZE_PER_GPU=$BATCH_SIZE_PER_GPU"
echo "GRADIENT_ACC_STEPS=$GRADIENT_ACC_STEPS"
echo "MAX_TRAIN_STEPS=$MAX_TRAIN_STEPS"
echo "MAX_SEQ_LENGTH=$MAX_SEQ_LENGTH"
echo "DEEPSPEED_CONFIG_FILE=$DEEPSPEED_CONFIG_FILE"
echo "PRECISION=$PRECISION"
echo "TOKENIZER_FLAGS=$TOKENIZER_FLAGS"
echo "DS_SKIP_CUDA_CHECK=$DS_SKIP_CUDA_CHECK"
echo "TORCH_EXTENSIONS_DIR=$TORCH_EXTENSIONS_DIR"
echo "SYNC_CHECKPOINT_TO_DRIVE=$SYNC_CHECKPOINT_TO_DRIVE"
echo "MIN_CHECKPOINT_MB=$MIN_CHECKPOINT_MB"
echo "EXTRA_ARGS=$EXTRA_ARGS"
echo "==================="

TOTAL_RAM_GB=$(python - <<'PY'
import psutil
print(int(psutil.virtual_memory().total/1024**3))
PY
)
if [ "$MODEL_KEY" = "mistral_7b" ] || [ "$MODEL_KEY" = "llama3_8b" ]; then
  if [ "$TOTAL_RAM_GB" -lt 150 ]; then
    echo "WARNING: 7B full-parameter training usually needs A100/High-RAM and may still saturate RAM."
  fi
fi

accelerate launch \
  --mixed_precision "$PRECISION" \
  --num_machines 1 \
  --num_processes "$NUM_GPUS" \
  --use_deepspeed \
  --deepspeed_config_file "$DEEPSPEED_CONFIG_FILE" \
  --main_process_port 8000 \
  fine_nwp/finetune.py \
  --model_name_or_path "$MODEL_NAME" \
  --tokenizer_name "$MODEL_NAME" \
  $TOKENIZER_FLAGS \
  --train_file "$TRAIN_FILE" \
  --max_seq_length "$MAX_SEQ_LENGTH" \
  --preprocessing_num_workers 2 \
  --per_device_train_batch_size "$BATCH_SIZE_PER_GPU" \
  --gradient_accumulation_steps "$GRADIENT_ACC_STEPS" \
  --learning_rate 2e-5 \
  --lr_scheduler_type linear \
  --warmup_ratio 0.03 \
  --weight_decay 0.0 \
  --num_train_epochs 3 \
  --max_train_steps "$MAX_TRAIN_STEPS" \
  --output_dir "$LOCAL_OUT" \
  --logging_steps 1 \
  --gradient_checkpointing \
  --overwrite_output_dir \
  $EXTRA_ARGS

echo "=== TRAIN FINISHED; validating local checkpoint ==="
find "$LOCAL_OUT" -maxdepth 3 -type f | head -80 || true
LOCAL_MB=$(du -sm "$LOCAL_OUT" | awk '{print $1}')
echo "LOCAL_CHECKPOINT_MB=$LOCAL_MB"
if [ "$LOCAL_MB" -lt "$MIN_CHECKPOINT_MB" ]; then
  echo "ERROR: local checkpoint is too small; save likely failed: $LOCAL_OUT"
  exit 4
fi

if [ "$SYNC_CHECKPOINT_TO_DRIVE" = "1" ]; then
  echo "=== SYNCING CHECKPOINT TO DRIVE ==="
  rm -rf "$DRIVE_OUT.tmp" "$DRIVE_OUT"
  mkdir -p "$DRIVE_OUT.tmp"
  rsync -ah --info=progress2 "$LOCAL_OUT/" "$DRIVE_OUT.tmp/"
  mv "$DRIVE_OUT.tmp" "$DRIVE_OUT"
  DRIVE_MB=$(du -sm "$DRIVE_OUT" | awk '{print $1}')
  echo "DRIVE_CHECKPOINT_MB=$DRIVE_MB"
  if [ "$DRIVE_MB" -lt "$MIN_CHECKPOINT_MB" ]; then
    echo "ERROR: Drive checkpoint is too small after sync: $DRIVE_OUT"
    exit 5
  fi
fi

if [ "$KEEP_LOCAL_CHECKPOINTS" != "1" ]; then
  rm -rf "$LOCAL_OUT"
fi

echo "=== TRAIN DONE ==="
echo "LOCAL_OUT=$LOCAL_OUT"
echo "DRIVE_OUT=$DRIVE_OUT"


Overwriting /content/drive/MyDrive/tow_project/scripts/tow_train_one.sh


In [12]:
%%writefile /content/drive/MyDrive/tow_project/scripts/tow_eval_one.sh
#!/usr/bin/env bash
set -euo pipefail

MODEL_KEY=${1:?MODEL_KEY required}
COND=${2:?COND required: Base, Raw, ToW, ToW-NoDeN, or ToW-PartDeN}
BENCH=${3:?BENCH required: arc-challenge, csqa, gsm8k, strategyqa, truthfulqa, halueval}
NUM_INSTANCES=${4:-}
PROJECT_DIR=${PROJECT_DIR:-/content/drive/MyDrive/tow_project}
REPO_DIR=${REPO_DIR:-/content/tow}
LOCAL_CHECKPOINT_ROOT=${LOCAL_CHECKPOINT_ROOT:-/content/tow_runtime/checkpoints}

cd "$REPO_DIR"

case "$MODEL_KEY" in
  qwen2_5_3b) BASE_MODEL="Qwen/Qwen2.5-3B" ;;
  phi3_mini) BASE_MODEL="microsoft/Phi-3-mini-4k-instruct" ;;
  tinyllama_1b) BASE_MODEL="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T" ;;
  mistral_7b) BASE_MODEL="mistralai/Mistral-7B-v0.1" ;;
  llama3_8b) BASE_MODEL="meta-llama/Meta-Llama-3-8B" ;;
  *) echo "Unknown MODEL_KEY: $MODEL_KEY"; exit 1 ;;
esac

checkpoint_valid() {
  local p="$1"
  [ -d "$p" ] || return 1
  local mb
  mb=$(du -sm "$p" 2>/dev/null | awk '{print $1}')
  MIN_CHECKPOINT_MB=${MIN_CHECKPOINT_MB:-500}
  [ -n "$mb" ] && [ "$mb" -ge "$MIN_CHECKPOINT_MB" ] || return 1
  return 0
}

if [ "$COND" = "Base" ]; then
  MODEL_PATH="$BASE_MODEL"
else
  LOCAL_PATH="$LOCAL_CHECKPOINT_ROOT/$MODEL_KEY/$COND"
  DRIVE_PATH="$PROJECT_DIR/checkpoints/$MODEL_KEY/$COND"
  if checkpoint_valid "$LOCAL_PATH"; then
    MODEL_PATH="$LOCAL_PATH"
  elif checkpoint_valid "$DRIVE_PATH"; then
    MODEL_PATH="$DRIVE_PATH"
  else
    echo "ERROR: no valid checkpoint found for $MODEL_KEY $COND"
    echo "Checked local: $LOCAL_PATH"
    echo "Checked Drive: $DRIVE_PATH"
    echo "Train it first with tow_train_one.sh, or use COND=Base for base-model eval."
    exit 2
  fi
fi

first_existing_dir() {
  for d in "$@"; do
    if [ -d "$d" ]; then echo "$d"; return 0; fi
  done
  return 1
}

case "$BENCH" in
  arc-challenge) DATA_DIR=$(first_existing_dir data/eval/arc_challenge data/eval/arc-challenge || true); MAXLEN=512 ;;
  csqa) DATA_DIR=$(first_existing_dir data/eval/csqa data/eval/commonsenseqa data/eval/common_sense_qa || true); MAXLEN=512 ;;
  gsm8k) DATA_DIR=$(first_existing_dir data/eval/gsm8k data/eval/gsm8k_main || true); MAXLEN=512 ;;
  strategyqa) DATA_DIR=$(first_existing_dir data/eval/strategyqa data/eval/strategy_qa || true); MAXLEN=512 ;;
  truthfulqa) DATA_DIR=$(first_existing_dir data/eval/truthfulqa data/eval/truthful_qa || true); MAXLEN=512 ;;
  halueval) DATA_DIR=$(first_existing_dir data/eval/halueval data/eval/halu_eval || true); MAXLEN=512 ;;
  *) echo "Unknown benchmark: $BENCH"; exit 1 ;;
esac

if [ -z "${DATA_DIR:-}" ] || [ ! -d "$DATA_DIR" ]; then
  echo "ERROR: evaluation data directory not found for BENCH=$BENCH"
  find data/eval -maxdepth 2 -type d | sort || true
  exit 3
fi

SAVE_DIR="$PROJECT_DIR/results/reproduction/$MODEL_KEY/$COND/$BENCH"
mkdir -p "$SAVE_DIR" "$PROJECT_DIR/logs"

EXTRA_ARGS=""
if [ -n "$NUM_INSTANCES" ]; then
  EXTRA_ARGS="--num_instances $NUM_INSTANCES"
fi

echo "=== EVAL START ==="
echo "MODEL_KEY=$MODEL_KEY"
echo "COND=$COND"
echo "BENCH=$BENCH"
echo "MODEL_PATH=$MODEL_PATH"
echo "DATA_DIR=$DATA_DIR"
echo "SAVE_DIR=$SAVE_DIR"
echo "NUM_INSTANCES=${NUM_INSTANCES:-FULL}"
echo "EXTRA_ARGS=$EXTRA_ARGS"
echo "=================="

CUDA_VISIBLE_DEVICES=0 python -m eval.run_eval \
  --eval_dataset "$BENCH" \
  --data_dir "$DATA_DIR" \
  --save_dir "$SAVE_DIR" \
  --model_name_or_path "$MODEL_PATH" \
  --tokenizer_name_or_path "$MODEL_PATH" \
  --newline_stop \
  --stop_at_triple_newline \
  --clm_max_length "$MAXLEN" \
  $EXTRA_ARGS

echo "=== EVAL DONE: $SAVE_DIR ==="

Overwriting /content/drive/MyDrive/tow_project/scripts/tow_eval_one.sh


In [13]:
%%writefile /content/drive/MyDrive/tow_project/scripts/tow_full_reproduction.sh
#!/usr/bin/env bash
set -euo pipefail

PROJECT_DIR=${PROJECT_DIR:-/content/drive/MyDrive/tow_project}
REPO_DIR=${REPO_DIR:-/content/tow}
RUN_MODE=${RUN_MODE:-smoke}
MODEL_KEYS_STR=${MODEL_KEYS_STR:-"qwen2_5_3b"}
BENCHMARKS_STR=${BENCHMARKS_STR:-"arc-challenge"}
CONDITIONS_STR=${CONDITIONS_STR:-"Raw ToW"}
LOCAL_CHECKPOINT_ROOT=${LOCAL_CHECKPOINT_ROOT:-/content/tow_runtime/checkpoints}
MIN_CHECKPOINT_MB=${MIN_CHECKPOINT_MB:-500}

TOTAL_BATCH_SIZE=${TOTAL_BATCH_SIZE:-4}
BATCH_SIZE_PER_GPU=${BATCH_SIZE_PER_GPU:-1}
MAX_TRAIN_STEPS=${MAX_TRAIN_STEPS:-2}
MAX_TRAIN_SAMPLES=${MAX_TRAIN_SAMPLES:-}
MAX_SEQ_LENGTH=${MAX_SEQ_LENGTH:-1024}
NUM_EVAL_INSTANCES=${NUM_EVAL_INSTANCES:-25}
DEEPSPEED_CONFIG_FILE=${DEEPSPEED_CONFIG_FILE:-configs/stage3_no_offloading_accelerate.conf}
SYNC_CHECKPOINT_TO_DRIVE=${SYNC_CHECKPOINT_TO_DRIVE:-1}
KEEP_LOCAL_CHECKPOINTS=${KEEP_LOCAL_CHECKPOINTS:-1}

export PROJECT_DIR REPO_DIR RUN_MODE TOTAL_BATCH_SIZE BATCH_SIZE_PER_GPU MAX_TRAIN_STEPS MAX_TRAIN_SAMPLES MAX_SEQ_LENGTH NUM_EVAL_INSTANCES DEEPSPEED_CONFIG_FILE LOCAL_CHECKPOINT_ROOT SYNC_CHECKPOINT_TO_DRIVE KEEP_LOCAL_CHECKPOINTS MIN_CHECKPOINT_MB
mkdir -p "$PROJECT_DIR/logs" "$PROJECT_DIR/results/reproduction" "$PROJECT_DIR/checkpoints"

IFS=' ' read -r -a MODELS <<< "$MODEL_KEYS_STR"
IFS=' ' read -r -a BENCHES <<< "$BENCHMARKS_STR"
IFS=' ' read -r -a CONDS <<< "$CONDITIONS_STR"
chmod +x "$PROJECT_DIR/scripts/tow_train_one.sh" "$PROJECT_DIR/scripts/tow_eval_one.sh"

valid_ckpt() {
  local p="$1"
  [ -d "$p" ] || return 1
  local mb
  mb=$(du -sm "$p" 2>/dev/null | awk '{print $1}')
  [ -n "$mb" ] && [ "$mb" -ge "$MIN_CHECKPOINT_MB" ] || return 1
  return 0
}

echo "=== REPRODUCTION DRIVER ==="
echo "RUN_MODE=$RUN_MODE"
echo "MODELS=${MODELS[*]}"
echo "CONDITIONS=${CONDS[*]}"
echo "BENCHMARKS=${BENCHES[*]}"
echo "MAX_TRAIN_STEPS=$MAX_TRAIN_STEPS"
echo "NUM_EVAL_INSTANCES=${NUM_EVAL_INSTANCES:-FULL}"
echo "LOCAL_CHECKPOINT_ROOT=$LOCAL_CHECKPOINT_ROOT"
echo "SYNC_CHECKPOINT_TO_DRIVE=$SYNC_CHECKPOINT_TO_DRIVE"
echo "==========================="

for MODEL in "${MODELS[@]}"; do
  for COND in "${CONDS[@]}"; do
    DRIVE_CKPT="$PROJECT_DIR/checkpoints/$MODEL/$COND"
    LOCAL_CKPT="$LOCAL_CHECKPOINT_ROOT/$MODEL/$COND"
    if valid_ckpt "$DRIVE_CKPT" || valid_ckpt "$LOCAL_CKPT"; then
      echo "Checkpoint exists; skipping train: $MODEL $COND"
    else
      LOG="$PROJECT_DIR/logs/train_${MODEL}_${COND}.log"
      echo "Launching train: $MODEL $COND"
      bash "$PROJECT_DIR/scripts/tow_train_one.sh" "$MODEL" "$COND" "$MAX_TRAIN_STEPS" 2>&1 | tee "$LOG"
    fi
  done
done

for MODEL in "${MODELS[@]}"; do
  for COND in "${CONDS[@]}"; do
    for BENCH in "${BENCHES[@]}"; do
      METRICS="$PROJECT_DIR/results/reproduction/$MODEL/$COND/$BENCH/metrics.json"
      if [ -f "$METRICS" ]; then
        echo "Metrics exist; skipping eval: $METRICS"
      else
        LOG="$PROJECT_DIR/logs/eval_${MODEL}_${COND}_${BENCH}.log"
        echo "Launching eval: $MODEL $COND $BENCH"
        bash "$PROJECT_DIR/scripts/tow_eval_one.sh" "$MODEL" "$COND" "$BENCH" "$NUM_EVAL_INSTANCES" 2>&1 | tee "$LOG"
      fi
    done
  done
done

python "$PROJECT_DIR/scripts/summarize_reproduction.py"


Overwriting /content/drive/MyDrive/tow_project/scripts/tow_full_reproduction.sh


In [14]:
%%writefile /content/drive/MyDrive/tow_project/scripts/summarize_reproduction.py
import json, csv, re
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/tow_project')
root = PROJECT_DIR / 'results' / 'reproduction'

def pick_metric(data):
    for key in ['exact_match', 'exact match', 'Exact match', 'accuracy', 'acc', 'score']:
        if key in data:
            return data.get(key)
    # Some scripts nest metrics under eval or results.
    for v in data.values():
        if isinstance(v, dict):
            got = pick_metric(v)
            if got is not None:
                return got
    return None

rows = []
for metrics in root.glob('*/*/*/metrics.json'):
    try:
        data = json.loads(metrics.read_text())
    except Exception as e:
        print('Could not read', metrics, e)
        continue
    model, cond, bench = metrics.parts[-4], metrics.parts[-3], metrics.parts[-2]
    rows.append({
        'model': model,
        'condition': cond,
        'benchmark': bench,
        'exact_match': pick_metric(data)
    })

rows = sorted(rows, key=lambda r: (r['model'], r['benchmark'], r['condition']))
out_csv = PROJECT_DIR / 'results' / 'reproduction_summary.csv'
out_json = PROJECT_DIR / 'results' / 'reproduction_summary.json'
out_csv.parent.mkdir(parents=True, exist_ok=True)

with out_csv.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['model','condition','benchmark','exact_match'])
    writer.writeheader()
    writer.writerows(rows)

out_json.write_text(json.dumps(rows, indent=2))
print('Wrote', out_csv)
print('Wrote', out_json)

if not rows:
    print('No metrics.json files found yet under', root)
else:
    for r in rows:
        print(r)


Overwriting /content/drive/MyDrive/tow_project/scripts/summarize_reproduction.py


In [15]:
!chmod +x /content/drive/MyDrive/tow_project/scripts/*.sh
!ls -lh /content/drive/MyDrive/tow_project/scripts


total 16K
-rw------- 1 root root 1.6K May  4 20:32 summarize_reproduction.py
-rwx------ 1 root root 3.5K May  4 20:32 tow_eval_one.sh
-rwx------ 1 root root 3.2K May  4 20:32 tow_full_reproduction.sh
-rwx------ 1 root root 6.3K May  4 20:32 tow_train_one.sh


## 8. Smoke test: base 3B model on ARC-Challenge


In [ ]:
# Smoke-test environment sanity check. This cell intentionally does NOT install anything.
# If these assertions fail, go back to the dependency cell; do not run ad-hoc pip installs here.
import numpy as np, transformers, tokenizers, vllm, peft
print("numpy:", np.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("vllm:", vllm.__version__)
print("peft:", peft.__version__)
assert np.__version__ == "1.26.4"
assert transformers.__version__ == "4.43.4"
assert tokenizers.__version__ == "0.19.1"
assert vllm.__version__ == "0.6.1.post2"
assert peft.__version__ == "0.12.0"
print("Smoke eval environment OK.")


numpy: 1.26.4
transformers: 4.43.4
tokenizers: 0.19.1
vllm: 0.6.1.post2
peft: 0.12.0
Smoke eval environment OK.


In [ ]:
%cd {REPO_DIR}
!bash /content/drive/MyDrive/tow_project/scripts/tow_eval_one.sh qwen2_5_3b Base arc-challenge 25   2>&1 | tee /content/drive/MyDrive/tow_project/logs/smoke_eval_qwen2_5_3b_base_arc.log


/content/tow
=== EVAL START ===
MODEL_KEY=qwen2_5_3b
COND=Base
BENCH=arc-challenge
MODEL_PATH=Qwen/Qwen2.5-3B
DATA_DIR=data/eval/arc_challenge
SAVE_DIR=/content/drive/MyDrive/tow_project/results/reproduction/qwen2_5_3b/Base/arc-challenge
NUM_INSTANCES=25
EXTRA_ARGS=--num_instances 25
Loading data...
Loading model and tokenizer...
INFO 05-02 06:32:26 llm_engine.py:223] Initializing an LLM engine (v0.6.1.post2) with config: model='Qwen/Qwen2.5-3B', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(gui

## 9. Run reproduction safely
Run the one-at-a-time cells first. The full driver is guarded and should stay off until the pair succeeds.


In [16]:
import os

if RUN_MODE == 'pilot':
    max_train_steps = str(PILOT_MAX_TRAIN_STEPS)
    max_train_samples = str(PILOT_MAX_TRAIN_SAMPLES)
    num_eval_instances = str(PILOT_NUM_EVAL_INSTANCES)
elif RUN_MODE == 'smoke':
    max_train_steps = '2'
    max_train_samples = '32'
    num_eval_instances = '25'
else:
    max_train_steps = str(FULL_MAX_TRAIN_STEPS)
    max_train_samples = str(FULL_MAX_TRAIN_SAMPLES)
    num_eval_instances = str(FULL_NUM_EVAL_INSTANCES)

os.environ['PROJECT_DIR'] = PROJECT_DIR
os.environ['REPO_DIR'] = REPO_DIR
os.environ['RUN_MODE'] = RUN_MODE
os.environ['MODEL_KEYS_STR'] = ' '.join(MODEL_KEYS)
os.environ['BENCHMARKS_STR'] = ' '.join(BENCHMARKS)
os.environ['CONDITIONS_STR'] = ' '.join(CONDITIONS)
os.environ['TOTAL_BATCH_SIZE'] = str(TOTAL_BATCH_SIZE)
os.environ['BATCH_SIZE_PER_GPU'] = str(BATCH_SIZE_PER_GPU)
os.environ['MAX_TRAIN_STEPS'] = max_train_steps
os.environ['MAX_TRAIN_SAMPLES'] = max_train_samples
os.environ['MAX_SEQ_LENGTH'] = str(MAX_SEQ_LENGTH)
os.environ['NUM_EVAL_INSTANCES'] = num_eval_instances
os.environ['DEEPSPEED_CONFIG_FILE'] = DEEPSPEED_CONFIG_FILE
os.environ['LOCAL_CHECKPOINT_ROOT'] = LOCAL_CHECKPOINT_ROOT
os.environ['LOCAL_TORCH_EXTENSIONS'] = LOCAL_TORCH_EXTENSIONS
os.environ['SYNC_CHECKPOINT_TO_DRIVE'] = SYNC_CHECKPOINT_TO_DRIVE
os.environ['KEEP_LOCAL_CHECKPOINTS'] = KEEP_LOCAL_CHECKPOINTS
os.environ['MIN_CHECKPOINT_MB'] = str(MIN_CHECKPOINT_MB)

print('RUN_MODE:', RUN_MODE)
print('MODEL_KEYS:', MODEL_KEYS)
print('BENCHMARKS:', BENCHMARKS)
print('CONDITIONS:', CONDITIONS)
print('TOTAL_BATCH_SIZE:', TOTAL_BATCH_SIZE)
print('BATCH_SIZE_PER_GPU:', BATCH_SIZE_PER_GPU)
print('MAX_TRAIN_STEPS:', max_train_steps)
print('MAX_TRAIN_SAMPLES:', max_train_samples if max_train_samples else 'FULL')
print('MAX_SEQ_LENGTH:', MAX_SEQ_LENGTH)
print('NUM_EVAL_INSTANCES:', num_eval_instances if num_eval_instances else 'FULL')
print('DEEPSPEED_CONFIG_FILE:', DEEPSPEED_CONFIG_FILE)
print('LOCAL_CHECKPOINT_ROOT:', LOCAL_CHECKPOINT_ROOT)
print('SYNC_CHECKPOINT_TO_DRIVE:', SYNC_CHECKPOINT_TO_DRIVE)
print('MIN_CHECKPOINT_MB:', MIN_CHECKPOINT_MB)


RUN_MODE: pilot
MODEL_KEYS: ['qwen2_5_3b']
BENCHMARKS: ['arc-challenge', 'csqa']
CONDITIONS: ['Raw', 'ToW']
TOTAL_BATCH_SIZE: 4
BATCH_SIZE_PER_GPU: 1
MAX_TRAIN_STEPS: 30
MAX_TRAIN_SAMPLES: 1024
MAX_SEQ_LENGTH: 768
NUM_EVAL_INSTANCES: 50
DEEPSPEED_CONFIG_FILE: configs/stage3_no_offloading_accelerate.conf
LOCAL_CHECKPOINT_ROOT: /content/tow_runtime/checkpoints
SYNC_CHECKPOINT_TO_DRIVE: 1
MIN_CHECKPOINT_MB: 500


In [17]:
# Final guard before launching reproduction. Do not install here; dependency pins should already be correct.
import transformers, peft, numpy as np, vllm
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("numpy:", np.__version__)
print("vllm:", vllm.__version__)
assert transformers.__version__ == "4.43.4"
assert peft.__version__ == "0.12.0"
assert np.__version__ == "1.26.4"
assert vllm.__version__ == "0.6.1.post2"
print("Reproduction environment guard OK.")


transformers: 4.43.4
peft: 0.12.0
numpy: 1.26.4
vllm: 0.6.1.post2
Reproduction environment guard OK.


### 9A. Optional throwaway train smoke
This proves the 3B training loop runs but does not sync a checkpoint to Drive. Skip it if you already saw Step 1 / Step 2 loss output.


In [ ]:
RUN_THROWAWAY_TRAIN_SMOKE = False

if RUN_THROWAWAY_TRAIN_SMOKE:
    import os, subprocess
    old_sync = os.environ.get("SYNC_CHECKPOINT_TO_DRIVE", "1")
    os.environ["SYNC_CHECKPOINT_TO_DRIVE"] = "0"
    cmd = (
        "cd /content/tow && "
        "bash /content/drive/MyDrive/tow_project/scripts/tow_train_one.sh qwen2_5_3b Raw 2 "
        "2>&1 | tee /content/drive/MyDrive/tow_project/logs/train_smoke_qwen2_5_3b_raw_local_only.log"
    )
    subprocess.run(cmd, shell=True, check=True)
    os.environ["SYNC_CHECKPOINT_TO_DRIVE"] = old_sync
else:
    print("Skipping throwaway training smoke.")


### 9B. Train Raw checkpoint


In [18]:
%cd {REPO_DIR}
!bash /content/drive/MyDrive/tow_project/scripts/tow_train_one.sh qwen2_5_3b Raw {max_train_steps} \
  2>&1 | tee /content/drive/MyDrive/tow_project/logs/train_qwen2_5_3b_Raw_retry.log


/content/tow
=== RUNTIME RESOURCE CHECK ===
Mon May  4 20:33:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             54W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+---

### 9C. Verify Raw checkpoint


In [19]:
!echo "Local Raw checkpoint:"
!du -sh /content/tow_runtime/checkpoints/qwen2_5_3b/Raw || true
!find /content/tow_runtime/checkpoints/qwen2_5_3b/Raw -maxdepth 2 -type f | head -50 || true

!echo "\nDrive Raw checkpoint:"
!du -sh /content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/Raw || true
!find /content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/Raw -maxdepth 2 -type f | head -50 || true

!echo "\nLast Raw log lines:"
!tail -120 /content/drive/MyDrive/tow_project/logs/train_qwen2_5_3b_Raw_retry.log || true


Local Raw checkpoint:
5.8G	/content/tow_runtime/checkpoints/qwen2_5_3b/Raw
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/merges.txt
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/config.json
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/pytorch_model.bin.index.json
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/generation_config.json
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/pytorch_model-00002-of-00002.bin
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/vocab.json
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/added_tokens.json
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/tokenizer_config.json
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/special_tokens_map.json
/content/tow_runtime/checkpoints/qwen2_5_3b/Raw/pytorch_model-00001-of-00002.bin
\nDrive Raw checkpoint:
5.8G	/content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/Raw
/content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/Raw/added_tokens.json
/content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/

### 9D. Train ToW checkpoint


In [20]:
%cd {REPO_DIR}
!bash /content/drive/MyDrive/tow_project/scripts/tow_train_one.sh qwen2_5_3b ToW {max_train_steps} \
  2>&1 | tee /content/drive/MyDrive/tow_project/logs/train_qwen2_5_3b_ToW_retry.log


/content/tow
=== RUNTIME RESOURCE CHECK ===
Mon May  4 20:37:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             55W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+---

### 9E. Verify ToW checkpoint


In [21]:
!echo "Local ToW checkpoint:"
!du -sh /content/tow_runtime/checkpoints/qwen2_5_3b/ToW || true
!find /content/tow_runtime/checkpoints/qwen2_5_3b/ToW -maxdepth 2 -type f | head -50 || true

!echo "\nDrive ToW checkpoint:"
!du -sh /content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/ToW || true
!find /content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/ToW -maxdepth 2 -type f | head -50 || true

!echo "\nLast ToW log lines:"
!tail -120 /content/drive/MyDrive/tow_project/logs/train_qwen2_5_3b_ToW_retry.log || true


Local ToW checkpoint:
5.8G	/content/tow_runtime/checkpoints/qwen2_5_3b/ToW
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/merges.txt
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/config.json
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/pytorch_model.bin.index.json
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/generation_config.json
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/pytorch_model-00002-of-00002.bin
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/vocab.json
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/added_tokens.json
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/tokenizer_config.json
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/special_tokens_map.json
/content/tow_runtime/checkpoints/qwen2_5_3b/ToW/pytorch_model-00001-of-00002.bin
\nDrive ToW checkpoint:
5.8G	/content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/ToW
/content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/ToW/added_tokens.json
/content/drive/MyDrive/tow_project/checkpoints/qwen2_5_3b/

### 9F. Evaluate Raw and ToW


In [22]:
%cd {REPO_DIR}
import subprocess

for bench in BENCHMARKS:
    for cond in ["Raw", "ToW"]:
        log_name = f"eval_qwen2_5_3b_{cond}_{bench}.log".replace("/", "_")
        cmd = (
            f"bash /content/drive/MyDrive/tow_project/scripts/tow_eval_one.sh "
            f"qwen2_5_3b {cond} {bench} {num_eval_instances} "
            f"2>&1 | tee /content/drive/MyDrive/tow_project/logs/{log_name}"
        )
        print("\n>>>", cmd)
        subprocess.run(cmd, shell=True, check=True)

subprocess.run("python /content/drive/MyDrive/tow_project/scripts/summarize_reproduction.py", shell=True, check=True)


/content/tow

>>> bash /content/drive/MyDrive/tow_project/scripts/tow_eval_one.sh qwen2_5_3b Raw arc-challenge 50 2>&1 | tee /content/drive/MyDrive/tow_project/logs/eval_qwen2_5_3b_Raw_arc-challenge.log

>>> bash /content/drive/MyDrive/tow_project/scripts/tow_eval_one.sh qwen2_5_3b ToW arc-challenge 50 2>&1 | tee /content/drive/MyDrive/tow_project/logs/eval_qwen2_5_3b_ToW_arc-challenge.log

>>> bash /content/drive/MyDrive/tow_project/scripts/tow_eval_one.sh qwen2_5_3b Raw csqa 50 2>&1 | tee /content/drive/MyDrive/tow_project/logs/eval_qwen2_5_3b_Raw_csqa.log

>>> bash /content/drive/MyDrive/tow_project/scripts/tow_eval_one.sh qwen2_5_3b ToW csqa 50 2>&1 | tee /content/drive/MyDrive/tow_project/logs/eval_qwen2_5_3b_ToW_csqa.log


CompletedProcess(args='python /content/drive/MyDrive/tow_project/scripts/summarize_reproduction.py', returncode=0)

In [ ]:
# Optional full driver. Leave this False until the one-at-a-time Raw/ToW flow works.
RUN_FULL_DRIVER = False

if RUN_FULL_DRIVER:
    import subprocess
    subprocess.run(
        "bash /content/drive/MyDrive/tow_project/scripts/tow_full_reproduction.sh "
        "2>&1 | tee /content/drive/MyDrive/tow_project/logs/full_reproduction_master.log",
        shell=True,
        check=True,
    )
else:
    print("Skipping full driver. Use the one-at-a-time cells above/below first.")


## 10. Monitor reproduction results

In [23]:
!echo 'Recent logs:'
!ls -lt /content/drive/MyDrive/tow_project/logs | head -20
!echo ''
!echo 'Current reproduction summary:'
!python /content/drive/MyDrive/tow_project/scripts/summarize_reproduction.py


Recent logs:
total 298
-rw------- 1 root root  4544 May  4 20:46 eval_qwen2_5_3b_ToW_csqa.log
-rw------- 1 root root  5754 May  4 20:45 eval_qwen2_5_3b_Raw_csqa.log
-rw------- 1 root root  4711 May  4 20:44 eval_qwen2_5_3b_ToW_arc-challenge.log
-rw------- 1 root root  5751 May  4 20:43 eval_qwen2_5_3b_Raw_arc-challenge.log
-rw------- 1 root root 62332 May  4 20:41 train_qwen2_5_3b_ToW_retry.log
-rw------- 1 root root 61571 May  4 20:37 train_qwen2_5_3b_Raw_retry.log
-rw------- 1 root root  5663 May  2 06:42 eval_qwen2_5_3b_tow_arc.log
-rw------- 1 root root  5390 May  2 06:41 eval_qwen2_5_3b_raw_arc.log
-rw------- 1 root root  4478 May  2 06:32 smoke_eval_qwen2_5_3b_base_arc.log
-rw------- 1 root root 68414 May  2 05:20 train_mistral_7b_Raw_retry.log
-rw------- 1 root root  5497 May  2 05:20 smoke_eval_mistral_base_arc.log
-rw------- 1 root root  5959 Apr 28 15:07 full_reproduction_master.log
-rw------- 1 root root  5690 Apr 28 15:07 train_mistral_7b_Raw.log
-rw------- 1 root root 5533

## 11. Files to use in the report

Main reproduction outputs:

- `/content/drive/MyDrive/tow_project/results/reproduction_summary.csv`
- `/content/drive/MyDrive/tow_project/results/reproduction_summary.json`
- `/content/drive/MyDrive/tow_project/results/reproduction/<model>/<condition>/<benchmark>/metrics.json`
- `/content/drive/MyDrive/tow_project/logs/train_<model>_<condition>.log`
- `/content/drive/MyDrive/tow_project/logs/eval_<model>_<condition>_<benchmark>.log`
- `/content/drive/MyDrive/tow_project/checkpoints/<model>/<condition>/`